# Baseline Comparison — HTD detectors

Pluggable framework. Each detector lives in its own dir and never knows whether its data is spatial / iid / synthetic. Two tracks:

- **EASY** — low-dimensional synthetic GMM (fast). i.i.d. and spatially-correlated. Sweeps target *model* (additive + **full replacement**), *signature* (comp_mean / between / orthogonal) and *amplitude*. Full-replacement (θ=1) is the *did-it-train* check.
- **HARD** — real Pavia scenarios (additive, amplitude sweep around 0.15).

Everything is saved per detector (model, scores, maps, metrics, train_log, config) so figures are remade WITHOUT retraining.

In [ ]:
# ── Cell 1: Clone/pull (branch iid-unified-experiment) + deps ─────────────
import os, sys
REPO_URL='https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL='/content/proj'; BRANCH='iid-unified-experiment'   # has all the latest code
if os.path.exists(os.path.join(LOCAL,'.git')):
    !git -C {LOCAL} fetch --all -q
    !git -C {LOCAL} checkout {BRANCH}
    !git -C {LOCAL} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {LOCAL}
!pip install -q scikit-learn scipy tqdm matplotlib pyyaml einops
for p in [LOCAL, os.path.join(LOCAL,'experiments','spatial')]:
    if p not in sys.path: sys.path.insert(0,p)
os.chdir(LOCAL); print('CWD', os.getcwd())
!git -C {LOCAL} log --oneline -1

In [ ]:
# ── Cell 2: results dir (+ optional Drive — only the HARD/Pavia run needs Drive) ──
import os
RESULTS = '/content/results'                       # default: local, no Drive needed
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/final_paper/baseline_results'   # persist to Drive
    os.makedirs('/content/proj/data', exist_ok=True)
    DST = '/content/proj/data/pavia-u.mat'
    if not os.path.exists(DST):
        for src in ['/content/drive/MyDrive/final_paper/pavia-u.mat',
                    '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat']:
            if os.path.exists(src):
                os.symlink(src, DST); print('linked', src); break
except Exception as e:
    print('Drive NOT mounted:', repr(e))
    print('-> using local RESULTS. EASY (synthetic) case works fully. '
          'HARD/Pavia needs Drive for pavia-u.mat — retry the mount for that.')
os.makedirs(RESULTS, exist_ok=True)
print('RESULTS =', RESULTS)

## EASY case — synthetic GMM (low-dim, fast)

In [ ]:
# ── Cell E1: EASY dry-run (quick sanity, both providers; no Drive needed) ──
import os; RESULTS = globals().get('RESULTS') or '/content/results'; os.makedirs(RESULTS, exist_ok=True)
CFG='experiments/baseline_comparison/configs/synthetic_gmm.yaml'
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider spatial_gmm --results_dir {RESULTS} --dry-run
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider iid_gmm     --results_dir {RESULTS} --dry-run

In [ ]:
# ── Cell E2: EASY full run (low-dim => fast; no Drive needed) ─────────────
# Scenarios + GMM data are DETERMINISTIC across runs, so you can REUSE models:
#   LOAD='all' or 'DSM,CF-Attn,CF-Attn-CFAR,NeighborMLP,MCLT'  -> load those, train the rest
#   PRETRAINED = a previous run dir to load from (e.g. the latest spatial_gmm_* run)
import os, glob
RESULTS = globals().get('RESULTS') or '/content/results'; os.makedirs(RESULTS, exist_ok=True)
CFG='experiments/baseline_comparison/configs/synthetic_gmm.yaml'
LOAD=''                      # '' = train everything fresh; else 'all' or 'A,B,...'
PRETRAINED=''                # e.g. sorted(glob.glob(f'{RESULTS}/spatial_gmm_*'))[-1]
EXTRA=f'--pretrained_dir {PRETRAINED} --load {LOAD}' if (LOAD and PRETRAINED) else ''
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider spatial_gmm --results_dir {RESULTS} {EXTRA}
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --provider iid_gmm     --results_dir {RESULTS} {EXTRA}

In [ ]:
# ── Cell E3: EASY figures (edit freely) ───────────────────────────────────
import os, glob, matplotlib.pyplot as plt
from experiments.baseline_comparison.framework import figures as F
RESULTS = globals().get('RESULTS') or '/content/results'
RUN = sorted(glob.glob(f'{RESULTS}/spatial_gmm_*') + glob.glob(f'{RESULTS}/dryrun_spatial_gmm_*'))[-1]
sc = F.scenarios(RUN)[0]
print('RUN', RUN, '| scenario', sc, '| detectors', F.detectors(RUN, sc))
F.summary_table(RUN, sc, 'additive', 'orthogonal')
for sig in ['comp_mean','between','orthogonal']:
    F.auc_vs_amplitude(RUN, sc, 'additive', sig); plt.show()
F.replacement_sanity(RUN, sc, 'orthogonal', 1.0); plt.show()   # did-it-train check
F.roc(RUN, sc, 'additive', 'orthogonal'); plt.show()
F.detection_maps(RUN, sc, 'additive', 'orthogonal'); plt.show()

## HARD case — Pavia scenarios (additive, amplitude sweep)

In [ ]:
# ── Cell H1: HARD dry-run (needs Drive for pavia-u.mat) ───────────────────
import os; RESULTS = globals().get('RESULTS') or '/content/results'; os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing — mount Drive (Cell 2) first'
CFG='experiments/baseline_comparison/configs/pavia_scenarios.yaml'
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --results_dir {RESULTS} --dry-run

In [ ]:
# ── Cell H2: HARD full run (Pavia, all detectors, additive + replacement sweep) ──
# Boxes/PCA are fixed across runs -> reuse expensive models, retrain only new ones:
#   LOAD='DSM,CF-Attn,CF-Attn-CFAR,NeighborMLP,MCLT'  PRETRAINED=<a previous pavia_* run>
import os, glob
RESULTS = globals().get('RESULTS') or '/content/results'; os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing — mount Drive (Cell 2) first'
CFG='experiments/baseline_comparison/configs/pavia_scenarios.yaml'
LOAD=''                      # '' = train everything fresh; else 'all' or 'A,B,...'
PRETRAINED=''                # e.g. sorted(glob.glob(f'{RESULTS}/pavia_*'))[-1]
EXTRA=f'--pretrained_dir {PRETRAINED} --load {LOAD}' if (LOAD and PRETRAINED) else ''
!python -u experiments/baseline_comparison/run_comparison.py --config {CFG} --results_dir {RESULTS} {EXTRA}

In [ ]:
# ── Cell H2b: HEAVY transductive baseline HTD-IRN (separate lean run) ──────
# HTD-IRN trains 5000 iters PER amplitude cell, so it gets its own short sweep on
# the SAME Pavia scenarios. Its run dir is MERGED with the main run in Cell H3.
import os; RESULTS = globals().get('RESULTS') or '/content/results'; os.makedirs(RESULTS, exist_ok=True)
assert os.path.exists('/content/proj/data/pavia-u.mat'), 'pavia-u.mat missing — mount Drive (Cell 2) first'
!python -u experiments/baseline_comparison/run_comparison.py \
    --config experiments/baseline_comparison/configs/htd_irn_pavia.yaml \
    --results_dir {RESULTS}

In [ ]:
# ── Cell H3: HARD figures (edit freely) — MERGES the main + HTD-IRN runs ───
import os, glob, matplotlib.pyplot as plt
from experiments.baseline_comparison.framework import figures as F
RESULTS = globals().get('RESULTS') or '/content/results'
# figures.* accept a LIST of run dirs and union their detectors. The main run and
# the HTD-IRN run are both named pavia_* ; the two latest are merged. Set RUNS
# explicitly, e.g. RUNS=['/.../pavia_<main_ts>', '/.../pavia_<htdirn_ts>'].
RUNS = sorted(glob.glob(f'{RESULTS}/pavia_*') + glob.glob(f'{RESULTS}/dryrun_pavia_*'))[-2:]
print('merging run dirs:', RUNS)
for sc in F.scenarios(RUNS):
    print('='*60); F.summary_table(RUNS, sc, 'additive')
    F.auc_vs_amplitude(RUNS, sc, 'additive'); plt.show()
    F.roc(RUNS, sc, 'additive', amplitude=0.15); plt.show()
    F.detection_maps(RUNS, sc, 'additive', amplitude=0.15); plt.show()